# PNADc Historical Proxy Engine v1.0.1\nCorreção: seleção de layout regular separada do seletor dos suplementos diretos.\n

# SPINE-GPE v7 — PNADc Historical Certification & Proxy Calibration Engine v1.0.1

Este notebook executa a auditoria, a calibração temporal 2022T4/2024T3 e a certificação das fontes regulares da PNADc já disponíveis no Drive.

**Limite obrigatório:** a série histórica recebe probabilidade modelada (`evidence tier C`); `platform_delivery_direct` permanece ausente.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPT = ROOT / 'scripts/SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.1.py'
REQ = ROOT / 'scripts/requirements_SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.1.txt'
UPSTREAM = ROOT / 'scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py'

print('ROOT:', ROOT)
print('SCRIPT:', SCRIPT, SCRIPT.exists())
print('REQ:', REQ, REQ.exists())
print('UPSTREAM:', UPSTREAM, UPSTREAM.exists())
assert SCRIPT.exists()
assert REQ.exists()
assert UPSTREAM.exists()


ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
SCRIPT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.1.py True
REQ: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/requirements_SPINE_GPEv7_PNADC_HISTORICAL_PROXY_ENGINE_v1.0.1.txt True
UPSTREAM: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNADC_CERTIFIER_v1.2.0.py True


In [3]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)
subprocess.run([sys.executable, '-m', 'py_compile', str(SCRIPT)], check=True)
print('py_compile: OK')


py_compile: OK


## 1. Auditoria das fontes locais

O modo `auto` descobre os trimestres já materializados em `data_pnadc` e `01_raw/10_ibge/pnadc_historical`, excluindo os módulos especiais diretos.


In [4]:
audit = subprocess.run(
    [
        sys.executable, str(SCRIPT),
        '--root', str(ROOT),
        '--mode', 'audit',
        '--periods', 'auto',
        '--strict',
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(audit.stdout)
print(audit.stderr)
print('Audit exit code:', audit.returncode)


2026-07-21 02:52:26,631 | INFO | SPINE-GPE PNADc Historical Proxy Engine v1.0.1 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=audit | periods=auto
2026-07-21 02:54:24,982 | INFO | Auditoria concluída | status=AUDIT_PASSED | períodos=['2022q4', '2024q3']
{
  "run_id": "20260721T025226Z",
  "script_version": "1.0.1",
  "validation_schema_version": "spine-gpe-v7-pnadc-historical-validation-1.0.1",
  "mode": "audit",
  "status": "AUDIT_PASSED",
  "critical_failures": [],
  "warnings": [],
  "requested_periods": [
    "2022q4",
    "2024q3"
  ],
  "sources": [
    {
      "period": "2022q4",
      "path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/data_pnadc/PNADC_042022.txt",
      "suffix": ".txt",
      "sha256": "4ae10a8879be31c0344f135aed645be87b0c0466538986093e0303c96485b6ab",
      "record_width": 3478,
      "layout_path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221

In [5]:
AUDIT_LOCK = ROOT / '00_admin/PNADC_HISTORICAL_PROXY_AUDIT_LOCK.json'
if not AUDIT_LOCK.exists():
    raise RuntimeError('Lock de auditoria não foi criado. Revise STDOUT/STDERR.')
audit_lock = json.loads(AUDIT_LOCK.read_text(encoding='utf-8'))
print(json.dumps(audit_lock, ensure_ascii=False, indent=2))
assert audit_lock['status'] == 'AUDIT_PASSED', audit_lock['critical_failures']
print('Períodos descobertos:', audit_lock['requested_periods'])


{
  "run_id": "20260721T025226Z",
  "script_version": "1.0.1",
  "validation_schema_version": "spine-gpe-v7-pnadc-historical-validation-1.0.1",
  "mode": "audit",
  "status": "AUDIT_PASSED",
  "critical_failures": [],
  "warnings": [],
  "requested_periods": [
    "2022q4",
    "2024q3"
  ],
  "sources": [
    {
      "period": "2022q4",
      "path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/data_pnadc/PNADC_042022.txt",
      "suffix": ".txt",
      "sha256": "4ae10a8879be31c0344f135aed645be87b0c0466538986093e0303c96485b6ab",
      "record_width": 3478,
      "layout_path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/02_interim/10_pnadc_certification/extracted_docs/Dicionario_e_input_20221031/dicionario_PNADC_microdados_trimestral.xls",
      "layout_sha256": "3f6b46ea00fd1549b52f878e3834135b6b2ae71a09ed576c458330e522e80eda",
      "layout_width": 3478,
      "source_kind": "regular_quarterly"
    },
    {
      "period": "2024q3",
      "path": "/content/driv

## 2. Calibração e certificação

Esta etapa:

1. lê os Parquets diretos certificados de 2022 e 2024;
2. valida temporalmente o modelo nos dois sentidos;
3. calibra a probabilidade;
4. aplica o modelo apenas aos trimestres regulares descobertos;
5. gera outputs imutáveis, model card, estimativas, lock e freeze.


In [6]:
full = subprocess.run(
    [
        sys.executable, str(SCRIPT),
        '--root', str(ROOT),
        '--mode', 'full',
        '--periods', 'auto',
        '--chunk-rows', '50000',
        '--strict',
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(full.stdout)
print(full.stderr)
print('Full exit code:', full.returncode)


2026-07-21 03:06:04,138 | INFO | SPINE-GPE PNADc Historical Proxy Engine v1.0.1 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=full | periods=auto
2026-07-21 03:07:28,561 | INFO | 2022q4: 50000 registros brutos lidos; 16916 no universo histórico
2026-07-21 03:07:38,456 | INFO | 2022q4: 100000 registros brutos lidos; 32003 no universo histórico
2026-07-21 03:07:46,142 | INFO | 2022q4: 150000 registros brutos lidos; 47361 no universo histórico
2026-07-21 03:07:56,190 | INFO | 2022q4: 200000 registros brutos lidos; 62280 no universo histórico
2026-07-21 03:08:04,264 | INFO | 2022q4: 250000 registros brutos lidos; 80351 no universo histórico
2026-07-21 03:08:14,162 | INFO | 2022q4: 300000 registros brutos lidos; 100296 no universo histórico
2026-07-21 03:08:24,496 | INFO | 2022q4: 350000 registros brutos lidos; 121877 no universo histórico
2026-07-21 03:08:32,498 | INFO | 2022q4: 400000 registros brutos lidos; 144150 no universo histórico
2026-07-21 03:08:44,551 | INFO

In [7]:
LOCK = ROOT / '00_admin/PNADC_HISTORICAL_PROXY_CERTIFICATION_LOCK.json'
if not LOCK.exists():
    raise RuntimeError('Lock de certificação não foi criado. Revise STDOUT/STDERR.')
lock = json.loads(LOCK.read_text(encoding='utf-8'))
print(json.dumps(lock, ensure_ascii=False, indent=2))
assert lock['status'] == 'CERTIFIED', lock['critical_failures']
print('STATUS:', lock['status'])
print('MODELO:', lock['model'])
print('PERÍODOS:', lock['certified_periods'])
print('REPORT:', lock['report'])


{
  "run_id": "20260721T030604Z",
  "script_version": "1.0.1",
  "schema_version": "spine-gpe-v7-pnadc-historical-proxy-1.0.0",
  "validation_schema_version": "spine-gpe-v7-pnadc-historical-validation-1.0.1",
  "model_schema_version": "spine-gpe-v7-pnadc-proxy-model-1.0.0",
  "mode": "full",
  "status": "CERTIFIED",
  "critical_failures": [],
  "warnings": [],
  "requested_periods": [
    "2022q4",
    "2024q3"
  ],
  "certified_periods": [
    "2022q4",
    "2024q3"
  ],
  "direct_inputs": {
    "2022": {
      "path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_processed/10_pnadc_certified/certified_pnadc_platform_2022.parquet",
      "sha256": "30dcfee62e19b17eb6a2e0526f3100424a60a3a3f7877870bdf1d1a081e876e2"
    },
    "2024": {
      "path": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_processed/10_pnadc_certified/certified_pnadc_platform_2024.parquet",
      "sha256": "0b228eeeb55653a9ebca54a4c5e9003af7be2ac7953a78241dc14d75ce18297c"
    }
  },
  "upstr

## 3. Inspeção das métricas temporais e estimativas


In [8]:
import pandas as pd

run_id = lock['run_id']
metrics_path = ROOT / f'05_outputs/tables/pnadc_historical_proxy/pnadc_proxy_temporal_metrics_{run_id}.csv'
rules_path = ROOT / f'05_outputs/tables/pnadc_historical_proxy/pnadc_proxy_rule_benchmark_{run_id}.csv'
estimates_path = Path(lock['estimates'])

metrics = pd.read_csv(metrics_path)
rules = pd.read_csv(rules_path)
estimates = pd.read_csv(estimates_path)

display(metrics)
display(rules)
display(estimates)


,model,train_year,test_year,n_test,n_positive,weighted_prevalence,roc_auc,average_precision,brier,null_brier,brier_skill,log_loss,ece_10
0,weighted_logit_occ_activity_position,2024,2022,178163,691,0.005208,0.980863,0.331738,0.003954,0.005181,0.236815,0.016074,5.287540e-04
1,weighted_logit_occ_activity_position,2022,2024,184157,778,0.005505,0.980062,0.374635,0.003986,0.005475,0.271932,0.016302,6.793733e-04
2,weighted_logit_occ_activity_position_calibrate...,0,0,362320,1469,0.005359,0.981253,0.349444,0.003916,0.005331,0.265323,0.015840,1.172943e-18
3,weighted_logit_extended_sensitivity,2024,2022,178163,691,0.005208,0.982918,0.372392,0.003912,0.005181,0.245040,0.015604,5.297994e-04
4,weighted_logit_extended_sensitivity,2022,2024,184157,778,0.005505,0.981921,0.389540,0.003968,0.005475,0.275281,0.015932,5.507342e-04
5,weighted_logit_extended_sensitivity_calibrated...,0,0,362320,1469,0.005359,0.982972,0.377126,0.003876,0.005331,0.272829,0.015465,2.150017e-18


,year,rule,n,n_positive,weighted_sensitivity,weighted_specificity,weighted_ppv,weighted_npv,weighted_accuracy
0,2022,occupation_compatible,178163,691,0.531010,0.938692,0.043379,0.997391,0.936568
1,2022,delivery_activity,178163,691,0.431591,0.997109,0.438705,0.997024,0.994164
2,2022,occupation_or_activity,178163,691,0.535719,0.938413,0.043557,0.997416,0.936316
3,2022,occupation_and_activity,178163,691,0.426882,0.997388,0.461083,0.997001,0.994416
4,2024,occupation_compatible,184157,778,0.561790,0.938108,0.047843,0.997421,0.936036
5,2024,delivery_activity,184157,778,0.459623,0.997019,0.460521,0.997009,0.994061
6,2024,occupation_or_activity,184157,778,0.563402,0.937908,0.047827,0.997430,0.935846
7,2024,occupation_and_activity,184157,778,0.458011,0.997219,0.476942,0.997000,0.994251


,period,estimand,total,total_se,share,share_se,n,n_eff,n_strata,n_psu
0,2022q4,model_expected_probability,4.544614e+05,13889.371916,0.005309,0.000158,178163,88610.681422,573,14960
1,2022q4,rule_occupation,5.457930e+06,82780.235247,0.063755,0.000880,178163,88610.681422,573,14960
2,2022q4,rule_activity,4.386371e+05,25628.432354,0.005124,0.000297,178163,88610.681422,573,14960
3,2022q4,rule_occ_or_activity,5.483772e+06,83297.973722,0.064057,0.000886,178163,88610.681422,573,14960
4,2022q4,rule_occ_and_activity,4.127947e+05,24541.341646,0.004822,0.000284,178163,88610.681422,573,14960
5,2022q4,model_class_threshold_0.25,3.316654e+05,22991.676694,0.003874,0.000267,178163,88610.681422,573,14960
6,2022q4,model_class_threshold_0.50,2.891664e+05,21833.221017,0.003378,0.000253,178163,88610.681422,573,14960
7,2022q4,model_class_threshold_0.75,0.000000e+00,0.000000,0.000000,0.000000,178163,88610.681422,573,14960
8,2024q3,model_expected_probability,4.884480e+05,13607.662481,0.005518,0.000151,184157,95984.942282,573,15070
9,2024q3,rule_occupation,5.721837e+06,82006.126871,0.064644,0.000860,184157,95984.942282,573,15070


## 4. Expansão histórica opcional

Execute somente depois que o modo `auto` estiver certificado. O comando abaixo pode baixar e processar muitos arquivos grandes.


In [9]:
# Exemplo controlado: período pandêmico e pré-módulo direto.

expansion = subprocess.run(
     [
         sys.executable, str(SCRIPT),
         '--root', str(ROOT),
         '--mode', 'full',
         '--periods', '2019q1:2021q4',
         '--download-missing',
         '--chunk-rows', '50000',
         '--strict',
     ],
     check=False,
 )


## Regras de uso

- A soma ponderada das probabilidades é o estimando principal.
- A classe binária é somente análise de sensibilidade.
- A série histórica não preenche `SD14001`, `S140093` ou `platform_delivery_direct`.
- Comparações entre períodos são descritivas/model-based, não um desenho causal.
